In [1]:
from pathways import Pathways
import bw2calc as bc
import pandas as pd
from bw2calc.graph_traversal import AssumedDiagonalGraphTraversal
from d3blocks import D3Blocks

fp = "/Users/romain/Github/sweet_sure-2050-switzerland/dev/remind-SSP2-NPi-stem-SPS1.zip"
p = Pathways(datapackage=fp, debug=True)

Invalid datapackage: Descriptor validation error: {'path': 'mapping/mapping.yaml', 'profile': 'data-resource', 'name': 'mapping', 'format': 'yaml', 'mediatype': 'text/yaml', 'encoding': 'utf-8'} is not valid under any of the given schemas at "resources/29" in descriptor and at "properties/resources/items/oneOf" in profile
Invalid datapackage: Descriptor validation error: 'data-resource' is not one of ['tabular-data-resource'] at "resources/29/profile" in descriptor and at "properties/resources/items/properties/profile/enum" in profile
Log file: /Users/romain/Library/Logs/pathways/pathways.log


In [5]:
sankey = p.sankey(
    #method='EF v3.0 EN15804 - climate change - global warming potential (GWP100)',
    method='ReCiPe 2016 v1.03, midpoint (H) - particulate matter formation - particulate matter formation potential (PMFP)',
    model="remind",
    scenario="SSP2-NPi-SPS1",
    region="CH",
    year=2020,
    #variable='FE_industry_process_heat_pump',
    variable=[v for v in p.scenarios.coords["variables"].values if "FE_cars" in v],
    cutoff=1e-3,
)

{37285: 740396690.1804485, 37284: 64767290.8777829, 37286: 991128.6127231001, 37282: 64143285485.75293, 37283: 75612915014.40279}


In [6]:
sankey.lca.demand

{37285: 740396690.1804485,
 37284: 64767290.8777829,
 37286: 991128.6127231001,
 37282: 64143285485.75293,
 37283: 75612915014.40279}

In [10]:
sum(sankey.lca.demand_array) == sum(sankey.lca.demand.values())

True

In [7]:
gt = AssumedDiagonalGraphTraversal().calculate(sankey.lca, max_calc=100)

/opt/homebrew/Caskroom/miniforge/base/envs/pathways/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:736: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 1.21e+13)
  warnings.warn(msg, UmfpackWarning)


100
0
2
4
5
39
52
62
67
71
94
108


/opt/homebrew/Caskroom/miniforge/base/envs/pathways/lib/python3.11/site-packages/bw2calc/graph_traversal.py:171: UserWarning: Stopping traversal due to calculation count.
  warnings.warn("Stopping traversal due to calculation count.")


In [11]:
df = pd.DataFrame(gt["edges"])

In [12]:
df.columns = ["target", "source", "amount", "exc_amount", "weight"]
df["unit"] = sankey.lcia_unit

In [13]:
df["target"] = df["target"].map({v: f"{k[0]} [{k[-1]}]" for k, v in sankey.activity_dict.items()})
df["source"] = df["source"].map({v: f"{k[0]} [{k[-1]}]" for k, v in sankey.activity_dict.items()})

In [14]:
df["target"] = df["target"].str.replace("market for", "m. for")
df["source"] = df["source"].str.replace("market for", "m. for")

In [15]:
len(df)

33

NameError: name 'bw2data' is not defined

In [16]:
d3_graph = D3Blocks()
d3_graph.sankey(
    df=df[1:],
    link={"color": "source-target"},
    filepath="/Users/romain/Github/pathways/dev/test.html",
)

In [25]:
p.scenarios.sel(
    year=2020,
    variables=[v for v in p.scenarios.coords["variables"].values if v.startswith("FE_cars")],
    region="CH"
).sum()*1e9

<xarray.DataArray 'value' ()> Size: 8B
array(1.40562356e+11)
Coordinates:
    region   <U2 8B 'CH'
    year     int64 8B 2020

In [26]:
p.scenarios.sel(
    year=2050,
    variables=[v for v in p.scenarios.coords["variables"].values if v.startswith("FE_cars")],
    region="CH"
).sum()*1e9

<xarray.DataArray 'value' ()> Size: 8B
array(4.62017741e+10)
Coordinates:
    region   <U2 8B 'CH'
    year     int64 8B 2050

In [1]:
import bw2data
from polyviz import sankey
bw2data.projects.set_current("ei310")

In [2]:
act = [a for a in bw2data.Database("2050 2") if "FE_" in a ["name"]][0]
act

'hydrogen burned, in hydrogen boiler_FE_residential_space_heating_hydrogen' (megajoule, CH, None)

In [3]:
sankey(
    activity=act,
    method=('IPCC 2021', 'climate change: fossil', 'global warming potential (GWP100)'),
    #level=3,
    cutoff=0.1,
)

Calculating supply chain score...


'/Users/romain/Github/sweet_sure-2050-switzerland/dev/hydrogen burned in hydrogen boilerFEresidentialspaceheatinghydrogen megajoule CH IPCC 2021climate change fossilglobal warming potential GWP100 sankey.html'